# Session 8: pandas

The same questions as the last two sessions, answered in Python.

Today should feel easy. Every idea here you already have from sessions 4, 6
and 7. Only the notation is new.

## Setup

In [ ]:
import os
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

import pandas as pd

pd.set_option("display.width", 110)
print("pandas", pd.__version__)

If that import failed, install it first:

```
pip install pandas matplotlib
```

`pip` downloads from **PyPI**, the Python Package Index: a public library of
code other people wrote. This is the thing that makes Python worth learning.
Somebody has already written the fiddly part of almost any job you have.

Right now you are installing into the Python for your whole machine, which is
fine for learning and becomes a problem once two projects want different
versions. **Session 10 fixes that properly** with virtual environments.

## Loading data: two ways in

In [ ]:
plays = pd.read_csv("data/clean/plays.csv", parse_dates=["played_at"])
plays.head()

Compare that one line with session 4: it replaces the `open`, the
`DictReader`, the `list()`, and **every single `float()`**. `read_csv` works
out the types, and `parse_dates` turns the date text into real dates you can
do arithmetic with.

The other way in is straight from the database.

In [ ]:
import sqlite3

con = sqlite3.connect("data/music.db")

db_plays = pd.read_sql("SELECT * FROM plays", con, parse_dates=["played_at"])
artists = pd.read_sql("SELECT * FROM artists", con)
awards = pd.read_sql("SELECT * FROM awards", con)

print(db_plays.shape, artists.shape, awards.shape)

`read_sql` takes **any** query, so everything you learned in the last two
sessions still applies. You can do the joining and grouping in SQL and hand
pandas a small tidy result, which is very often the best of both.

In [ ]:
pd.read_sql("""
    SELECT device, COUNT(*) AS plays, ROUND(AVG(minutes_played), 2) AS avg_min
    FROM plays
    GROUP BY device
    ORDER BY plays DESC
""", con)

## The first five minutes with any table

Make this a ritual: `shape`, `info`, `head`, `describe`, in that order, on
every dataset you meet for the rest of your life.

In [ ]:
plays.shape          # rows, columns

In [ ]:
plays.dtypes

Every column has a name **and a type**. That is the difference from your list
of dictionaries in session 4: pandas knows `4.65` is a number and
`played_at` is a date.

(Text columns show as `str` in pandas 3 and as `object` in pandas 2. Same
thing, and the older name is still what most tutorials say.)

In [ ]:
plays.info()

`.info()` is the one to run first on data you have not seen. It shows the
type of every column **and how many values are missing**, which is the first
question worth asking. It is also where session 9 begins.

In [ ]:
plays.describe().round(2)

Six lines and you know the shape of the data: 2,183 plays, averaging 4.11
minutes, nothing longer than 11, about 12% skipped. Getting that in session 4
took a page of Python.

## Selecting columns

In [ ]:
plays["genre"].head()          # one column is a Series

In [ ]:
plays[["genre", "minutes_played"]].head()      # note the DOUBLE brackets

One column on its own is a **Series**. Two or more is a **DataFrame**. That
is why the double brackets: the inner pair is a *list* of the columns you
want.

In [ ]:
print(plays.loc[0, "genre"])
plays.loc[0:2, ["genre", "device"]]

`.loc` is **label based**: rows by index label, columns by name. And unlike
everything else in Python, `.loc[0:2]` **includes** row 2. It is the one
place slicing is inclusive, and it catches everybody once.

There is also `.iloc`, which is purely positional and exclusive at the end.
It exists; use `.loc`.

## Filtering rows

This is the one with the sharp edge.

In [ ]:
long = plays[plays["minutes_played"] > 8]
len(long)

Two conditions need `&` and `|`, and **every condition needs brackets**.

In [ ]:
recent_long = plays[
    (plays["minutes_played"] > 8)
    & (plays["played_at"] >= "2025-01-01")
]
len(recent_long)

Now the mistake everybody makes. Uncomment it and read the message.

In [ ]:
# plays[plays["minutes_played"] > 8 and plays["device"] == "phone"]
# ValueError: The truth value of a Series is ambiguous.

Python's `and` wants **one** true-or-false value. Here each side is 2,183 of
them, so pandas refuses rather than guessing. That is what "the truth value
of a Series is ambiguous" means.

And the brackets matter because `&` binds tighter than `>`, so without them
pandas reads the expression in the wrong order and the error is baffling.

`isin` is the equivalent of SQL's `IN`:

In [ ]:
len(plays[plays["device"].isin(["car", "tablet"])])

## Sorting and counting

In [ ]:
plays.sort_values("minutes_played", ascending=False).head(5)

In [ ]:
plays["device"].value_counts()

**That is your session 4 counting dictionary and session 6's
`GROUP BY device`, in one method call.** Same idea, third notation.

In [ ]:
plays["device"].value_counts(normalize=True).round(3)

`normalize=True` gives proportions instead of counts, which saves the
division and the percentage mistake that goes with it.

## groupby: the same idea again

In [ ]:
plays.groupby("genre")["minutes_played"].sum().round(1)

In [ ]:
(plays.groupby("genre")["minutes_played"]
      .agg(["count", "sum", "mean"])
      .round(2)
      .sort_values("sum", ascending=False))

Read that left to right as a sentence: **split by genre, take the minutes
column, work these out for each group.**

Jazz has the highest average play again, at 6.57 minutes. Same finding as
sessions 4 and 7, reached a third way.

The named form is the closest match to SQL's `AS`, and it is the one to
prefer once you want different functions on different columns:

In [ ]:
plays.groupby("device").agg(
    plays=("play_id", "count"),
    minutes=("minutes_played", "sum"),
    avg=("minutes_played", "mean"),
).round(2).sort_values("plays", ascending=False)

## The index surprise

This confuses everybody once, so let us get it out of the way.

In [ ]:
by_genre = plays.groupby("genre")["minutes_played"].sum()
print(by_genre["Jazz"])          # works
print(by_genre.index[:3])        # genre is the INDEX now

In [ ]:
# by_genre["genre"]
# KeyError: 'genre'

The grouping key moves out of the columns and becomes the **row labels**.
Everything that then treats it as a column fails, including plotting and
writing to CSV in the shape you expected.

The fix is one method:

In [ ]:
by_genre = (plays.groupby("genre")["minutes_played"]
                 .sum()
                 .reset_index())
print(list(by_genre.columns))
by_genre.head(3)

`reset_index()` pushes the index back into a proper column and numbers the
rows again. Reach for it whenever a groupby result is awkward to work with:
it is usually the answer.

## merge is JOIN

In [ ]:
both = db_plays.merge(artists, on="artist_id")
print("inner:", len(both))

left = artists.merge(db_plays, on="artist_id", how="left")
print("left: ", len(left))

2,183 against 2,185. The two extra rows are the artists who have never been
played, which the inner join dropped.

`NaN` is pandas' `NULL`, and `.isna()` is its `IS NULL`.

In [ ]:
left[left["play_id"].isna()][["artist_name", "genre", "country"]]

`how=` is the join type: `"inner"` by default, then `"left"`, `"right"` and
`"outer"`. The words mean exactly what they meant in SQL.

Now the useful shape, join then group:

In [ ]:
(both.groupby("artist_name")["minutes_played"]
     .sum()
     .round(1)
     .nlargest(5))

## Fan-out is not a SQL problem

In [ ]:
print("true total:  ", round(db_plays["minutes_played"].sum(), 1))

fan = db_plays.merge(awards, on="artist_id")
print("rows before: ", len(db_plays))
print("rows after:  ", len(fan))
print("merged total:", round(fan["minutes_played"].sum(), 1))

8,980.4 becomes 9,732.1. **Identical numbers to session 7**, because it is
the same arithmetic. Nothing about fan-out was specific to SQL.

pandas does give you one thing SQL does not, and hardly anybody uses it.

In [ ]:
print("plays keys unique? ", db_plays["artist_id"].is_unique)
print("awards keys unique?", awards["artist_id"].is_unique)

In [ ]:
try:
    db_plays.merge(awards, on="artist_id", validate="many_to_one")
except Exception as error:
    print(type(error).__name__, ":", error)

`validate=` makes you **state the shape you expect**, and complains when you
are wrong. Get in the habit: it turns a silent wrong answer into an error
message.

The options are `"one_to_one"`, `"one_to_many"`, `"many_to_one"` and
`"many_to_many"`. Here the awards side has repeated keys, so `many_to_one`
is a lie and pandas says so.

## Tidying up

In [ ]:
con.close()
print("closed")

## Next

`02-sql-and-pandas.ipynb` puts the two side by side, four questions at a
time.